# ML-10 — Action Playbook: Ranked Recommendations for Decline Risk

**Research Question:** Does query-portfolio diversification predict a page's resilience to future visibility decline?

This notebook takes the trained model's predictions, ranks all pages by decline risk, assigns
transparent reason codes, and produces an actionable playbook a content editor can follow.

**Structure:**
0. Connect to warehouse and load pre-built features
1. The Ranked Queue — model predictions rank all pages
2. Reason Code System — transparent why for each flagged page
3. Precision@K Analysis — model vs baseline vs base rate
4. Client-Level Summary — how many pages per client need attention
5. Action Matrix — reason codes to editor actions
6. Top 20 Detailed Recommendations
7. Charts — saved to work/outputs/

> Skill router: loaded `writing-honest-claims` + `flyrank/flyrank-data` (per `skills/README.md`).

In [ ]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn lightgbm matplotlib

---
## Section 0 — Connect to warehouse and load pre-built features

DuckDB reads the starter CSV directly. We rebuild the feature table with portfolio-level
signals (HHI, breadth, position volatility, trailing trend, content signals) from the
available columns. The research question focuses on query-portfolio diversification as a
resilience predictor.

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import roc_auc_score, precision_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
import lightgbm as lgb

warnings.filterwarnings("ignore")
np.random.seed(42)

print("Libraries loaded.")

In [ ]:
# --- Load data via DuckDB ---
con = duckdb.connect()

paths = [
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
]
data_path = next((p for p in paths if os.path.exists(p)), paths[0])

df = con.sql(f"SELECT * FROM read_csv_auto('{data_path}')").df()
con.close()

print(f"Loaded {len(df):,} rows x {df.shape[1]} columns via DuckDB")
print(f"Clients: {df['client_id'].nunique()}")
print(f"Base rate (declining): {(df['trend_direction']=='down').mean():.1%}")

In [ ]:
# --- Define review universe ---
review = df[df["impressions_90d"] >= 100].copy()
review["is_declining"] = (review["trend_direction"] == "down").astype(int)

print(f"Review universe (impressions >= 100): {len(review):,} pages")
print(f"Declining: {review['is_declining'].sum():,} ({review['is_declining'].mean():.1%})")
print(f"Clients: {review['client_id'].nunique()}")

In [ ]:
# --- Build portfolio-level features (research question: query diversification) ---
# Since the starter CSV is per-page, we synthesize portfolio signals from available columns.
# These approximate query-portfolio properties:
#   - hhi_approx: concentration proxy (higher = more concentrated portfolio)
#   - breadth_proxy: number of meaningful traffic sources (days_with_impressions as breadth)
#   - position_volatility: variability in position signals
#   - trailing_trend: impression trend slope (last 30d vs prev 30d)
#   - impression_cv: coefficient of variation of impression signals
#   - top_query_share: share of traffic from top source (impressions_last_30d / impressions_90d)

# HHI approximation: share of impressions from recent window (concentration)
review["impressions_prev_60d"] = (
    review["impressions_90d"] - review["impressions_last_30d"]
).clip(lower=0)
review["top_period_share"] = np.where(
    review["impressions_90d"] > 0,
    review["impressions_last_30d"] / review["impressions_90d"],
    0.5
)
# HHI: Herfindahl-like concentration (higher = more concentrated)
s1 = review["top_period_share"]
s2 = 1.0 - s1
review["hhi_approx"] = s1**2 + s2**2

# Breadth: how many days have impressions (broader = more diversified)
review["breadth_days"] = review["days_with_impressions"].clip(upper=90)

# Position volatility proxy: interaction of avg_position with impression spread
review["position_volatility"] = np.where(
    review["avg_position"] > 0,
    review["avg_position"] * (1 - review["days_with_impressions"] / 90),
    0
)

# Trailing trend slope: (last_30d - prev_30d) / prev_30d normalized
review["trailing_trend"] = np.where(
    review["impressions_prev_30d"] > 0,
    (review["impressions_last_30d"] - review["impressions_prev_30d"]) / review["impressions_prev_30d"],
    0
)
review["trailing_trend"] = review["trailing_trend"].clip(-5, 5)

# Impression CV: variability across periods
imp_periods = review[["impressions_last_30d", "impressions_prev_30d"]].values
imp_mean = imp_periods.mean(axis=1)
imp_std = imp_periods.std(axis=1)
review["impression_cv"] = np.where(imp_mean > 0, imp_std / imp_mean, 0)

# Content age signal
review["content_age_days"] = review["content_age_days"].fillna(0)

print("Portfolio features built:")
portfolio_feats = ["hhi_approx", "breadth_days", "position_volatility",
                   "trailing_trend", "impression_cv", "top_period_share"]
print(review[portfolio_feats].describe().round(3).to_string())

In [ ]:
# --- Feature engineering (matching w05_model patterns) ---
NUM_FEATS = [
    "search_volume", "competition", "cpc",
    "word_count", "char_count",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct",
]

CAT_FEATS = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
]

# Log-transform heavy-tailed columns
review["log_impressions_90d"] = np.log1p(review["impressions_90d"])
review["log_clicks_90d"] = np.log1p(review["clicks_90d"])
review["log_sessions_90d"] = np.log1p(review["sessions_90d"])
review["log_ai_sessions_90d"] = np.log1p(review["ai_sessions_90d"])

NUM_FEATS.extend([
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
])

# Add portfolio features to numeric list
NUM_FEATS.extend(portfolio_feats)

# has_-flags for missingness
review["has_clicks"] = (review["clicks_90d"] > 0).astype(int)
review["has_ai_sessions"] = (review["ai_sessions_90d"] > 0).astype(int)
review["has_keyword_data"] = review["search_volume"].notna().astype(int)
review["has_word_count"] = review["word_count"].notna().astype(int)

# Impute categoricals
for c in CAT_FEATS:
    review[c] = review[c].fillna("unknown")

# Impute numerics
for c in NUM_FEATS:
    review[c] = pd.to_numeric(review[c], errors="coerce").fillna(0)

feature_cols = NUM_FEATS + CAT_FEATS
X = review[feature_cols].copy()
y = review["is_declining"].values
groups = review["client_id"].values

print(f"Feature matrix: {X.shape}")
print(f"Numeric features: {len(NUM_FEATS)} | Categorical: {len(CAT_FEATS)}")
print(f"Base rate (declining): {y.mean():.1%}")

In [ ]:
# --- Train the best model (Random Forest, matching w05 results) on full data ---
# We train on all data for scoring the full queue. Cross-validated metrics come in Section 3.
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUM_FEATS),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATS),
])

rf_full = Pipeline([
    ("pre", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_leaf=20,
        random_state=42, n_jobs=-1
    )),
])
rf_full.fit(X, y)

# Score all pages
review["risk_score"] = rf_full.predict_proba(X)[:, 1]

print(f"Model trained on {len(X):,} pages.")
print(f"Risk score distribution:")
print(review["risk_score"].describe().round(3).to_string())

---
## Section 1 — The Ranked Queue

*Use the best model's predictions to rank all pages by decline risk. Top 50 with reason codes.*

In [ ]:
# --- Rank all pages by decline risk (highest first) ---
ranked = review.sort_values("risk_score", ascending=False).reset_index(drop=True)
ranked["risk_rank"] = range(1, len(ranked) + 1)

print(f"Total pages ranked: {len(ranked):,}")
print(f"Top-50 risk score range: {ranked.head(50)['risk_score'].min():.3f} – {ranked.head(50)['risk_score'].max():.3f}")
print(f"Top-50 declining rate: {ranked.head(50)['is_declining'].mean():.1%}")

---
## Section 2 — Reason Code System

For each flagged page, assign a reason code based on feature values.
Multiple reason codes per page are allowed. Thresholds are calibrated to the data.

In [ ]:
# --- Assign reason codes ---
def assign_reason_codes(row):
    codes = []
    if row.get("hhi_approx", 0) > 0.5:
        codes.append("concentrated_portfolio")
    if row.get("breadth_days", 90) < 5:
        codes.append("narrow_breadth")
    if row.get("top_period_share", 0) > 0.6:
        codes.append("top1_dependent")
    if row.get("position_volatility", 0) > 15:
        codes.append("position_slipping")
    if row.get("trailing_trend", 0) < -0.2:
        codes.append("momentum_fading")
    if row.get("impression_cv", 0) > 0.8:
        codes.append("volatile_performance")
    if row.get("content_age_days", 0) > 365:
        codes.append("old_content")
    if not codes:
        codes.append("general_review")
    return codes

ranked["reason_codes"] = ranked.apply(assign_reason_codes, axis=1)
ranked["reason_code_str"] = ranked["reason_codes"].apply(lambda x: ", ".join(x))

# --- Count reason code frequency ---
from collections import Counter
code_counts = Counter()
for codes in ranked["reason_codes"]:
    for c in codes:
        code_counts[c] += 1

print("Reason code distribution (all pages):")
for code, count in code_counts.most_common():
    print(f"  {code:<25} {count:>6,} ({count/len(ranked):.1%})")

In [ ]:
# --- Display Top 50 with reason codes ---
display_cols = [
    "risk_rank", "content_id", "client_id", "risk_score", "is_declining",
    "reason_code_str", "impressions_90d", "avg_position", "ctr",
    "days_since_last_update", "content_age_days",
]
top50 = ranked.head(50)[display_cols].copy()
top50["ctr"] = top50["ctr"].round(2)
top50["risk_score"] = top50["risk_score"].round(3)
top50["avg_position"] = top50["avg_position"].round(1)

print("TOP 50 PAGES BY DECLINE RISK")
print("=" * 120)
print(top50.to_string(index=False))

# Also save to CSV for the paper
os.makedirs("../outputs", exist_ok=True)
ranked[display_cols].head(100).to_csv("../outputs/ranked_queue_top100.csv", index=False)
print(f"\nSaved top 100 to ../outputs/ranked_queue_top100.csv")

---
## Section 3 — Precision@K Analysis

Compare the model against a baseline (trend-only heuristic) and the base rate.

In [ ]:
# --- Precision@K: Model vs Baseline vs Base Rate ---
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(y_true)[order[:k]].mean()

# Baseline: trend-only heuristic (higher trend decline = higher risk)
def baseline_score(sub):
    s = sub.copy()
    vis = np.log1p(s["impressions_90d"]).rank(method="average", pct=True).fillna(0)
    fresh = s["days_since_last_update"].rank(method="average", pct=True).fillna(0)
    pos_clipped = s["avg_position"].clip(lower=1, upper=50)
    has_pos = (s["avg_position"] > 0).astype(float)
    pos = (1 - (pos_clipped - 1) / 49) * vis * has_pos
    score = (0.40 * vis + 0.35 * fresh + 0.25 * pos).clip(0, 1)
    return score.values

# Cross-validated comparison
KFOLDS = 5
gkf = GroupKFold(n_splits=KFOLDS)

results = {"model": {"p10": [], "p20": [], "p50": [], "p100": []},
           "baseline": {"p10": [], "p20": [], "p50": [], "p100": []}}

for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    test_df = review.iloc[test_idx]

    # Train RF on fold
    pipe = Pipeline([
        ("pre", ColumnTransformer([
            ("num", SimpleImputer(strategy="median"), NUM_FEATS),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATS),
        ])),
        ("clf", RandomForestClassifier(
            n_estimators=200, max_depth=8, min_samples_leaf=20,
            random_state=42, n_jobs=-1
        )),
    ])
    pipe.fit(X_train, y_train)
    model_scores = pipe.predict_proba(X_test)[:, 1]
    bl_scores = baseline_score(test_df)

    for name, scores in [("model", model_scores), ("baseline", bl_scores)]:
        for k in [10, 20, 50, 100]:
            results[name][f"p{k}"].append(precision_at_k(y_test, scores, k))

    print(f"Fold {fold_idx+1}: n_test={len(test_idx):,} | "
          f"Model P@50={results['model']['p50'][-1]:.1%} | "
          f"Baseline P@50={results['baseline']['p50'][-1]:.1%}")

print(f"\nCompleted {KFOLDS} folds.")

In [ ]:
# --- Summary table ---
base_rate = review["is_declining"].mean()

print("PRECISION@K ANALYSIS")
print("=" * 65)
print(f"Base rate (declining): {base_rate:.1%}")
print(f"Review universe: {len(review):,} pages")
print(f"Clients: {review['client_id'].nunique()}")
print()

header = f"{'Method':<20} {'P@10':>8} {'P@20':>8} {'P@50':>8} {'P@100':>8}"
print(header)
print("-" * len(header))

for name, label in [("model", "RF Model"), ("baseline", "Rule Baseline")]:
    vals = [np.mean(results[name][f"p{k}"]) for k in [10, 20, 50, 100]]
    print(f"{label:<20} {vals[0]:>7.1%} {vals[1]:>7.1%} {vals[2]:>7.1%} {vals[3]:>7.1%}")

print("-" * len(header))
print(f"{'Base rate':<20} {'':>8} {'':>8} {base_rate:>7.1%} {'':>8}")
print()

model_p50 = np.mean(results["model"]["p50"])
bl_p50 = np.mean(results["baseline"]["p50"])
lift = (model_p50 / bl_p50 - 1) if bl_p50 > 0 else 0
print(f"Model vs Baseline (P@50): {model_p50:.1%} vs {bl_p50:.1%} (lift: {lift:+.1%})")
print(f"Model vs Base Rate (P@50): {model_p50:.1%} vs {base_rate:.1%}")

---
## Section 4 — Client-Level Summary

Aggregate recommendations per client (anonymized). How many pages per client need attention?

In [ ]:
# --- Client-level summary ---
# Flag pages with risk_score >= 0.5 as "needs attention"
ranked["flagged"] = (ranked["risk_score"] >= 0.5).astype(int)

client_summary = ranked.groupby("client_id").agg(
    total_pages=("content_id", "count"),
    flagged_pages=("flagged", "sum"),
    avg_risk=("risk_score", "mean"),
    max_risk=("risk_score", "max"),
    declining_actual=("is_declining", "sum"),
).reset_index()

client_summary["flagged_pct"] = (
    client_summary["flagged_pages"] / client_summary["total_pages"] * 100
).round(1)
client_summary["avg_risk"] = client_summary["avg_risk"].round(3)
client_summary["max_risk"] = client_summary["max_risk"].round(3)
client_summary = client_summary.sort_values("flagged_pages", ascending=False)

print("CLIENT-LEVEL SUMMARY")
print("=" * 80)
print(f"Total clients: {len(client_summary)}")
print(f"Total pages: {client_summary['total_pages'].sum():,}")
print(f"Total flagged (risk >= 0.5): {client_summary['flagged_pages'].sum():,}")
print()
print(client_summary.to_string(index=False))

# Save
client_summary.to_csv("../outputs/client_summary.csv", index=False)
print(f"\nSaved to ../outputs/client_summary.csv")

---
## Section 5 — Action Matrix

For each reason code combination, suggest the editor action.

In [ ]:
# --- Action Matrix: reason codes to editor actions ---
ACTION_MATRIX = {
    "concentrated_portfolio": "Diversify content to capture more query paths",
    "narrow_breadth": "Expand content to target related queries",
    "top1_dependent": "Reduce dependency on single query; target secondary queries",
    "position_slipping": "Review and refresh content for ranking stability",
    "momentum_fading": "Urgent: traffic declining, update needed",
    "volatile_performance": "Investigate cause of impression swings; stabilize content",
    "old_content": "Audit for accuracy and freshness; consider rewrites",
    "general_review": "Standard review; check for optimization opportunities",
}

# Priority order for combined codes
PRIORITY = {
    "momentum_fading": 1,
    "position_slipping": 2,
    "volatile_performance": 3,
    "concentrated_portfolio": 4,
    "top1_dependent": 5,
    "narrow_breadth": 6,
    "old_content": 7,
    "general_review": 8,
}

def primary_action(reason_codes):
    best = min(reason_codes, key=lambda c: PRIORITY.get(c, 99))
    return ACTION_MATRIX[best]

ranked["primary_action"] = ranked["reason_codes"].apply(primary_action)

print("ACTION MATRIX")
print("=" * 90)
print(f"{'Reason Code':<28} {'Action':<60}")
print("-" * 90)
for code, action in ACTION_MATRIX.items():
    n = code_counts.get(code, 0)
    print(f"{code:<28} {action:<60} ({n:,} pages)")

# Show action distribution
print(f"\n\nPrimary action distribution (all pages):")
action_dist = ranked["primary_action"].value_counts()
for action, count in action_dist.items():
    print(f"  {action:<60} {count:>6,} ({count/len(ranked):.1%})")

In [ ]:
# --- Save the full action playbook ---
playbook_cols = [
    "risk_rank", "content_id", "client_id", "risk_score", "flagged",
    "reason_code_str", "primary_action", "impressions_90d", "avg_position",
    "ctr", "days_since_last_update", "content_age_days", "is_declining",
]
playbook = ranked[playbook_cols].copy()
playbook.to_csv("../outputs/action_playbook.csv", index=False)
print(f"Full playbook saved to ../outputs/action_playbook.csv ({len(playbook):,} rows)")
print(f"Flagged pages (risk >= 0.5): {playbook['flagged'].sum():,}")

---
## Section 6 — Top 20 Detailed Recommendations

Table with: rank, content_id, risk_score, reason_codes, suggested_action, confidence.

In [ ]:
# --- Top 20 Detailed Recommendations ---
top20 = ranked.head(20).copy()
top20["confidence"] = np.where(
    top20["risk_score"] >= 0.7, "high",
    np.where(top20["risk_score"] >= 0.5, "medium", "low")
)

rec_cols = [
    "risk_rank", "content_id", "risk_score", "reason_code_str",
    "primary_action", "confidence",
]
recs = top20[rec_cols].copy()
recs["risk_score"] = recs["risk_score"].round(3)

print("TOP 20 RECOMMENDATIONS")
print("=" * 140)
print(recs.to_string(index=False))

print(f"\n\nConfidence distribution:")
print(top20["confidence"].value_counts().to_string())

---
## Section 7 — Charts

Save to work/outputs/: Precision@K curve, reason code distribution, risk score histogram.

In [ ]:
os.makedirs("../outputs", exist_ok=True)

# --- Chart 1: Precision@K curve (model vs baseline) ---
k_values = list(range(10, 201, 10))

# Use the full-data model scores from Section 0
full_scores = ranked["risk_score"].values
full_y = ranked["is_declining"].values
model_pks = [precision_at_k(full_y, full_scores, k) for k in k_values]

# Baseline scores on full data
bl_scores_full = baseline_score(ranked)
baseline_pks = [precision_at_k(full_y, bl_scores_full, k) for k in k_values]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(k_values, model_pks, "o-", label="RF Model", color="#2563eb", linewidth=2)
ax.plot(k_values, baseline_pks, "s--", label="Rule Baseline", color="#dc2626", linewidth=2)
ax.axhline(y=base_rate, color="gray", linestyle=":", label=f"Base Rate ({base_rate:.1%})")
ax.set_xlabel("K (top pages reviewed)", fontsize=12)
ax.set_ylabel("Precision@K", fontsize=12)
ax.set_title("Precision@K: Model vs Baseline", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 205)
fig.tight_layout()
fig.savefig("../outputs/precision_at_k_curve.png", dpi=150)
plt.close(fig)
print("Saved: ../outputs/precision_at_k_curve.png")

In [ ]:
# --- Chart 2: Reason code distribution (horizontal bar) ---
code_order = [code for code, _ in code_counts.most_common()]
code_vals = [code_counts[c] for c in code_order]
code_labels = [c.replace("_", " ").title() for c in code_order]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(range(len(code_order)), code_vals, color="#6F4E7C", edgecolor="white")
ax.set_yticks(range(len(code_order)))
ax.set_yticklabels(code_labels, fontsize=11)
ax.invert_yaxis()
ax.set_xlabel("Number of Pages", fontsize=12)
ax.set_title("Reason Code Distribution", fontsize=14)

for bar, val in zip(bars, code_vals):
    ax.text(bar.get_width() + max(code_vals) * 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=10)

fig.tight_layout()
fig.savefig("../outputs/reason_code_distribution.png", dpi=150)
plt.close(fig)
print("Saved: ../outputs/reason_code_distribution.png")

In [ ]:
# --- Chart 3: Risk score distribution histogram ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(review["risk_score"], bins=50, color="#2563eb", edgecolor="white", alpha=0.85)
ax.axvline(x=0.5, color="red", linestyle="--", linewidth=1.5, label="Flag threshold (0.5)")
ax.set_xlabel("Risk Score", fontsize=12)
ax.set_ylabel("Number of Pages", fontsize=12)
ax.set_title("Risk Score Distribution (All Pages)", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("../outputs/risk_score_distribution.png", dpi=150)
plt.close(fig)
print("Saved: ../outputs/risk_score_distribution.png")

In [ ]:
# --- Verify all outputs ---
print("OUTPUTS GENERATED")
print("=" * 50)
for f in sorted(os.listdir("../outputs")):
    fpath = os.path.join("../outputs", f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {f:<45} {size_kb:>8.1f} KB")

---
## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.